<h1>TRAIN MODEL</h1>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import optuna
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

In [ ]:
X, y = load_digits(return_X_y=True)

In [ ]:
print(X.shape, y.shape)

In [ ]:
X[0]

In [ ]:
i = 7
plt.matshow(X[i].reshape(8, 8), cmap="grey")
print(y[i])
plt.show()

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

In [ ]:
from students.sandanov.lesson3 import Exercise

In [ ]:
X_train = X_train / 16.0
X_val = X_val / 16.0
X_test = X_test / 16.0

In [ ]:
numpy_model = Exercise.create_model(
    Exercise.create_linear_layer(64, 128),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(128, 64),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(64, 10),
    Exercise.create_logsoftmax_layer(),
)

In [ ]:
loss = Exercise.create_nll_loss()

Exercise.train_model(numpy_model, loss, X_train, y_train, 0.08087446569278885, 81, 16)

In [ ]:
val_predictions = numpy_model.forward(X_val)

In [ ]:
val_predicted_classes = np.argmax(val_predictions, axis=1)

In [ ]:
log_probs = numpy_model.forward(X_test)
yres = np.argmax(log_probs, axis=1)
metric = np.mean(yres == y_test)

print(f"Точность: {metric * 100:.2f}%")

In [ ]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    n_epoch = trial.suggest_int("n_epoch", 10, 100)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    hidden_dim = trial.suggest_int("hidden_dim", 32, 256)

    numpy_model = Exercise.create_model(
        Exercise.create_linear_layer(64, hidden_dim),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_dim, 64),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(64, 10),
        Exercise.create_logsoftmax_layer(),
    )

    loss_fn = Exercise.create_nll_loss()

    Exercise.train_model(numpy_model, loss_fn, X_train, y_train, lr=lr, n_epoch=n_epoch, batch_size=batch_size)

    val_preds = numpy_model.forward(X_val)
    accuracy = np.mean(np.argmax(val_preds, axis=1) == y_val)

    return accuracy


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Лучшие параметры:", study.best_params)
print("Лучшая точность:", study.best_value)